# Single Commodity Time Series Forecasting Model

**Objective:** Build and evaluate optimized forecasting models for individual commodity groups.

**Approach:**
- Focus on one commodity at a time
- Tune models based on specific commodity characteristics
- Generate 3-18 month forecasts with confidence intervals
- Document methodology for thesis

**Available Commodities:**
- wood, paper, chemicals, buildingmaterials, energy

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from contextlib import contextmanager
import sys
from io import StringIO

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
import pmdarima as pm
from prophet import Prophet

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Configuration & Data Loading

Load and prepare commodity data for forecasting analysis.

In [ ]:
# Configuration
COMMODITY = 'wood'
DATA_FOLDER = '/Users/Hans/Documents/EMBA/MasterThesis/Data'
FORECAST_WEEKS = 26
TEST_SIZE = 52

def convert_week_to_datetime(year, week):
    try:
        date_str = f"{year}-W{int(week):02d}-1"
        return pd.to_datetime(date_str, format='%G-W%V-%u')
    except:
        return pd.NaT

def load_commodity_data(commodity_name):
    possible_names = [f'{commodity_name.lower()}2005_2024.csv', f'{commodity_name.lower()}.csv']
    filepath = None
    for name in possible_names:
        full_path = os.path.join(DATA_FOLDER, name)
        if os.path.exists(full_path):
            filepath = full_path
            break
    if filepath is None:
        raise FileNotFoundError(f"No file found for {commodity_name}")
    
    df = pd.read_csv(filepath, sep=',')
    df.columns = df.columns.str.upper().str.strip()
    df_agg = df.groupby(['VJA', 'VMO', 'VKW']).agg({'TTOKM': 'sum'}).reset_index()
    df_agg.columns = ['year', 'month', 'week', 'ttokm']
    df_agg['date'] = df_agg.apply(lambda row: convert_week_to_datetime(int(row['year']), int(row['week'])), axis=1)
    df_agg = df_agg.sort_values('date').reset_index(drop=True)
    df_agg = df_agg[['date', 'year', 'month', 'week', 'ttokm']].copy()
    df_agg['ttokm'] = pd.to_numeric(df_agg['ttokm'], errors='coerce')
    return df_agg

df = load_commodity_data(COMMODITY)
print(f"✓ {COMMODITY.upper()}: {len(df)} weeks | {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Mean: {df['ttokm'].mean():.0f}, Std: {df['ttokm'].std():.0f}, Min: {df['ttokm'].min():.0f}, Max: {df['ttokm'].max():.0f}")

## 2. Exploratory Data Analysis

Analyze temporal patterns and distributions.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(df['date'], df['ttokm'], linewidth=1, color='steelblue', alpha=0.8)
axes[0].set_title(f'{COMMODITY.upper()} - Time Series', fontsize=12, fontweight='bold')
axes[0].set_ylabel('TTOKM')
axes[0].grid(True, alpha=0.3)

axes[1].hist(df['ttokm'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_title(f'{COMMODITY.upper()} - Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('TTOKM')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Skewness: {df['ttokm'].skew():.3f}, Kurtosis: {df['ttokm'].kurtosis():.3f}")

## 3. Stationarity Testing

Check if differencing is needed using ADF test.

In [ ]:
result = adfuller(df['ttokm'].dropna(), autolag='AIC')
print(f"ADF Statistic: {result[0]:.6f}, P-Value: {result[1]:.6f}")
print(f"Stationary: {'YES' if result[1] <= 0.05 else 'NO (d=1 recommended)'}")

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(df['date'], df['ttokm'], linewidth=1, color='steelblue')
axes[0].set_title(f'{COMMODITY.upper()} - Original', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)

first_diff = df['ttokm'].diff().dropna()
axes[1].plot(df['date'][1:], first_diff, linewidth=1, color='darkgreen')
axes[1].set_title(f'{COMMODITY.upper()} - First Difference', fontsize=11, fontweight='bold')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Train/Test Split

Prepare 95.8% training and 4.2% test data.

In [ ]:
train_df = df[:-TEST_SIZE].copy()
test_df = df[-TEST_SIZE:].copy()

print(f"Total: {len(df)} weeks | Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%) | Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_df['date'], train_df['ttokm'], label='Training', linewidth=1, color='steelblue', alpha=0.8)
ax.plot(test_df['date'], test_df['ttokm'], label='Test', linewidth=1, color='darkred', alpha=0.8)
ax.axvline(x=test_df['date'].iloc[0], color='black', linestyle='--', linewidth=2, label='Split')
ax.set_title(f'{COMMODITY.upper()} - Train/Test Split', fontsize=12, fontweight='bold')
ax.set_ylabel('TTOKM')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Prophet Model

Forecast using Facebook Prophet with confidence intervals.

In [ ]:
from prophet import Prophet

prophet_df = train_df[['date', 'ttokm']].copy()
prophet_df.columns = ['ds', 'y']

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True, 
                           changepoint_prior_scale=0.05, seasonality_prior_scale=10,
                           seasonality_mode='additive')
    prophet_model.fit(prophet_df)

future_test = prophet_model.make_future_dataframe(periods=TEST_SIZE, freq='W')
prophet_forecast_test = prophet_model.predict(future_test)
prophet_pred_test = prophet_forecast_test[-TEST_SIZE:][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].reset_index(drop=True)

prophet_rmse = np.sqrt(mean_squared_error(test_df['ttokm'].values, prophet_pred_test['yhat'].values))
prophet_mae = mean_absolute_error(test_df['ttokm'].values, prophet_pred_test['yhat'].values)
prophet_mape = np.mean(np.abs((test_df['ttokm'].values - prophet_pred_test['yhat'].values) / test_df['ttokm'].values)) * 100

print(f"Prophet - RMSE: {prophet_rmse:.2f}, MAE: {prophet_mae:.2f}, MAPE: {prophet_mape:.2f}%")

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(train_df['date'], train_df['ttokm'], label='Training', linewidth=1, color='steelblue', alpha=0.7)
ax.plot(test_df['date'], test_df['ttokm'], label='Actual', linewidth=1, color='darkred', alpha=0.8)
ax.plot(prophet_pred_test['ds'].dt.date, prophet_pred_test['yhat'], label='Prophet', linewidth=2, color='green')
ax.fill_between(prophet_pred_test['ds'].dt.date, prophet_pred_test['yhat_lower'], prophet_pred_test['yhat_upper'],
                 alpha=0.2, color='green', label='95% CI')
ax.set_title(f'{COMMODITY.upper()} - Prophet Forecast', fontsize=12, fontweight='bold')
ax.set_ylabel('TTOKM')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. LSTM Model

Deep learning forecast with LSTM layers.

In [ ]:
lookback = 52
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(train_df['ttokm'].values.reshape(-1, 1))

X_train_keras, y_train_keras = [], []
for i in range(lookback, len(scaled_data)):
    X_train_keras.append(scaled_data[i-lookback:i, 0])
    y_train_keras.append(scaled_data[i, 0])

X_train_keras = np.array(X_train_keras).reshape(-1, lookback, 1)
y_train_keras = np.array(y_train_keras)

tf.random.set_seed(42)
np.random.seed(42)

lstm_model = Sequential([
    LSTM(50, activation='relu', input_shape=(lookback, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(50, activation='relu'),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = lstm_model.fit(X_train_keras, y_train_keras, epochs=50, batch_size=32, verbose=0, validation_split=0.1)

test_data_scaled = scaler.transform(test_df['ttokm'].values.reshape(-1, 1))
combined_data = np.concatenate([scaled_data, test_data_scaled], axis=0)

X_test_keras = [combined_data[i-lookback:i, 0] for i in range(len(train_df), len(combined_data))]
X_test_keras = np.array(X_test_keras).reshape(-1, lookback, 1)

lstm_pred_scaled = lstm_model.predict(X_test_keras, verbose=0)
lstm_pred_test = scaler.inverse_transform(lstm_pred_scaled)[:TEST_SIZE]

lstm_rmse = np.sqrt(mean_squared_error(test_df['ttokm'].values, lstm_pred_test))
lstm_mae = mean_absolute_error(test_df['ttokm'].values, lstm_pred_test)
lstm_mape = np.mean(np.abs((test_df['ttokm'].values - lstm_pred_test.flatten()) / test_df['ttokm'].values)) * 100

print(f"LSTM - RMSE: {lstm_rmse:.2f}, MAE: {lstm_mae:.2f}, MAPE: {lstm_mape:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train', linewidth=1)
axes[0].plot(history.history['val_loss'], label='Val', linewidth=1)
axes[0].set_title(f'{COMMODITY.upper()} - LSTM Training', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_df['date'], train_df['ttokm'], label='Train', linewidth=1, color='steelblue', alpha=0.7)
axes[1].plot(test_df['date'], test_df['ttokm'], label='Actual', linewidth=1, color='darkred', alpha=0.8)
axes[1].plot(test_df['date'], lstm_pred_test, label='LSTM', linewidth=2, color='purple')
axes[1].set_title(f'{COMMODITY.upper()} - LSTM Forecast', fontsize=11, fontweight='bold')
axes[1].set_ylabel('TTOKM')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 18-Month Forward Forecast

Generate forward-looking forecast on full dataset.

In [ ]:
full_prophet_df = df[['date', 'ttokm']].copy()
full_prophet_df.columns = ['ds', 'y']

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    full_prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                                changepoint_prior_scale=0.05, seasonality_prior_scale=10,
                                seasonality_mode='additive')
    full_prophet_model.fit(full_prophet_df)

future_dates = full_prophet_model.make_future_dataframe(periods=FORECAST_WEEKS, freq='W')
forward_forecast = full_prophet_model.predict(future_dates)
forward_forecast = forward_forecast[-FORECAST_WEEKS:][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].reset_index(drop=True)
forward_forecast.columns = ['date', 'forecast', 'lower_bound', 'upper_bound']
forward_forecast['date'] = forward_forecast['date'].dt.strftime('%Y-%m-%d')

print(f"\nForecast: {FORECAST_WEEKS} weeks ({FORECAST_WEEKS/4.33:.1f} months)")
print(f"Period: {forward_forecast['date'].iloc[0]} to {forward_forecast['date'].iloc[-1]}")
print(f"Mean: {forward_forecast['forecast'].mean():.0f} | Range: [{forward_forecast['lower_bound'].min():.0f}, {forward_forecast['upper_bound'].max():.0f}]")
print(f"\nFirst 5 weeks:")
print(forward_forecast.head(5).to_string(index=False))

## 8. Export Results

Save forecasts and metrics to CSV.

In [ ]:
output_dir = f'./outputs/{COMMODITY}'
os.makedirs(output_dir, exist_ok=True)

forward_forecast.to_csv(f'{output_dir}/forecast_{FORECAST_WEEKS}weeks.csv', index=False)

metrics_df = pd.DataFrame({
    'Model': ['Prophet', 'LSTM'],
    'RMSE': [prophet_rmse, lstm_rmse],
    'MAE': [prophet_mae, lstm_mae],
    'MAPE_%': [prophet_mape, lstm_mape]
})
metrics_df.to_csv(f'{output_dir}/metrics.csv', index=False)

commodity_info = pd.DataFrame({
    'Commodity': [COMMODITY],
    'Total_Weeks': [len(df)],
    'Train_Weeks': [len(train_df)],
    'Test_Weeks': [len(test_df)],
    'Forecast_Weeks': [FORECAST_WEEKS],
    'Date_Start': [df['date'].min().strftime('%Y-%m-%d')],
    'Date_End': [df['date'].max().strftime('%Y-%m-%d')]
})
commodity_info.to_csv(f'{output_dir}/info.csv', index=False)

print(f"\n✓ Results exported to: {output_dir}/")
print(f"  - forecast_{FORECAST_WEEKS}weeks.csv")
print(f"  - metrics.csv")
print(f"  - info.csv")

# Single Commodity Time Series Forecasting Model

**Objective:** Build and evaluate optimized forecasting models for individual commodity groups.

**Approach:**
- Focus on one commodity at a time
- Tune models based on specific commodity characteristics
- Generate 3-18 month forecasts with confidence intervals
- Document methodology for thesis

**Available Commodities:**
- wood
- paper
- chemicals
- buildingmaterials
- energy

In [ ]:
%pip install -q numpy pandas matplotlib seaborn statsmodels pmdarima prophet tensorflow scikit-learn pyarrow

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from contextlib import contextmanager
import sys
from io import StringIO

# Time Series Libraries
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
import pmdarima as pm
from prophet import Prophet

# Machine Learning
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Select Commodity & Load Data

Choose the commodity to analyze and load historical data.

In [ ]:
# ===== CONFIGURATION =====
# SELECT COMMODITY HERE
COMMODITY = 'wood'  # Change to: 'paper', 'chemicals', 'buildingmaterials', 'energy'

# Data folder
DATA_FOLDER = '/Users/Hans/Documents/EMBA/MasterThesis/Data'

# Forecast horizon (weeks)
FORECAST_WEEKS = 26  # ~6 months (adjust for 3-18 month range: 13-78 weeks)

# Test set size (weeks)
TEST_SIZE = 52  # 1 year for validation

print(f"Commodity: {COMMODITY.upper()}")
print(f"Forecast horizon: {FORECAST_WEEKS} weeks (~{FORECAST_WEEKS/4.33:.1f} months)")
print(f"Test set: {TEST_SIZE} weeks")

# ===== DATA LOADING FUNCTIONS =====
def convert_week_to_datetime(year, week):
    """Convert ISO year and week to datetime"""
    try:
        date_str = f"{year}-W{int(week):02d}-1"
        return pd.to_datetime(date_str, format='%G-W%V-%u')
    except:
        return pd.NaT

def load_commodity_data(commodity_name):
    """Load and prepare commodity CSV"""
    possible_names = [
        f'{commodity_name.lower()}2005_2024.csv',
        f'{commodity_name.lower()}.csv',
    ]
    
    filepath = None
    for name in possible_names:
        full_path = os.path.join(DATA_FOLDER, name)
        if os.path.exists(full_path):
            filepath = full_path
            break
    
    if filepath is None:
        raise FileNotFoundError(f"No file found for {commodity_name}")
    
    # Load CSV
    df = pd.read_csv(filepath, sep=',')
    df.columns = df.columns.str.upper().str.strip()
    
    # Aggregate by (year, month, week) - sum across product codes
    df_agg = df.groupby(['VJA', 'VMO', 'VKW']).agg({
        'TTOKM': 'sum'
    }).reset_index()
    
    # Rename to standard format
    df_agg.columns = ['year', 'month', 'week', 'ttokm']
    
    # Create datetime
    df_agg['date'] = df_agg.apply(
        lambda row: convert_week_to_datetime(int(row['year']), int(row['week'])),
        axis=1
    )
    
    # Sort and clean
    df_agg = df_agg.sort_values('date').reset_index(drop=True)
    df_agg = df_agg[['date', 'year', 'month', 'week', 'ttokm']].copy()
    df_agg['ttokm'] = pd.to_numeric(df_agg['ttokm'], errors='coerce')
    
    return df_agg

# Load data
df = load_commodity_data(COMMODITY)
print(f"\n✓ Data loaded: {len(df)} weeks | {df['date'].min().date()} to {df['date'].max().date()}")

## 2. Exploratory Data Analysis

Understand the commodity's temporal patterns and characteristics.

In [ ]:
# Load commodity data
df = load_commodity_data(COMMODITY)
print(f"\n{'='*60}")
print(f"COMMODITY: {COMMODITY.upper()}")
print(f"{'='*60}")
print(f"Data Shape: {df.shape}")
print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
print(f"Duration: {(df['date'].max() - df['date'].min()).days / 365.25:.1f} years")
print(f"\nBasic Statistics (TTOKM):")
print(df['TTOKM'].describe())
print(f"\nMissing Values: {df['TTOKM'].isna().sum()}")

# Visualize full time series
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full time series
axes[0].plot(df['date'], df['TTOKM'], linewidth=1, color='steelblue', alpha=0.8)
axes[0].set_title(f'{COMMODITY.upper()} - Full Time Series (Weekly TTOKM)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('TTOKM (thousand ton-km)', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Distribution histogram
axes[1].hist(df['TTOKM'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_title(f'{COMMODITY.upper()} - Distribution of Weekly TTOKM Values', fontsize=12, fontweight='bold')
axes[1].set_xlabel('TTOKM (thousand ton-km)', fontsize=10)
axes[1].set_ylabel('Frequency', fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nSkewness: {df['TTOKM'].skew():.3f}")
print(f"Kurtosis: {df['TTOKM'].kurtosis():.3f}")

In [ ]:
# Train/Test Split
train_df = df[:-TEST_SIZE].copy()
test_df = df[-TEST_SIZE:].copy()

print(f"\nTrain/Test Split:")
print(f"{'='*50}")
print(f"Total weeks: {len(df)}")
print(f"Training weeks: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Testing weeks: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
print(f"\nTrain period: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"Test period: {test_df['date'].min()} to {test_df['date'].max()}")

# Visualize train/test split
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(train_df['date'], train_df['TTOKM'], label='Training Data', linewidth=1, color='steelblue', alpha=0.8)
ax.plot(test_df['date'], test_df['TTOKM'], label='Test Data', linewidth=1, color='darkred', alpha=0.8)

# Add vertical line at split point
split_date = test_df['date'].iloc[0]
ax.axvline(x=split_date, color='black', linestyle='--', linewidth=2, label='Train/Test Split')

ax.set_title(f'{COMMODITY.upper()} - Train/Test Split (52-week validation window)', fontsize=12, fontweight='bold')
ax.set_xlabel('Date', fontsize=10)
ax.set_ylabel('TTOKM (thousand ton-km)', fontsize=10)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Prepare LSTM data
lookback = 52  # 52-week lookback window

# Scale data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(train_df['TTOKM'].values.reshape(-1, 1))

# Create sequences for LSTM
X_train_keras = []
y_train_keras = []

for i in range(lookback, len(scaled_data)):
    X_train_keras.append(scaled_data[i-lookback:i, 0])
    y_train_keras.append(scaled_data[i, 0])

X_train_keras = np.array(X_train_keras).reshape(-1, lookback, 1)
y_train_keras = np.array(y_train_keras)

# Build LSTM model
tf.random.set_seed(42)
np.random.seed(42)

lstm_model = Sequential([
    LSTM(50, activation='relu', input_shape=(lookback, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(50, activation='relu'),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train LSTM
print(f"\nLSTM Model - Training...")
print(f"{'='*50}")
history = lstm_model.fit(
    X_train_keras, y_train_keras,
    epochs=50,
    batch_size=32,
    verbose=0,
    validation_split=0.1
)

print(f"Final training loss: {history.history['loss'][-1]:.6f}")

# Prepare scaled test data for prediction
test_data_scaled = scaler.transform(test_df['TTOKM'].values.reshape(-1, 1))

# Combine train + test for creating LSTM sequences
combined_data = np.concatenate([scaled_data, test_data_scaled], axis=0)

X_test_keras = []
for i in range(len(train_df), len(combined_data)):
    X_test_keras.append(combined_data[i-lookback:i, 0])

X_test_keras = np.array(X_test_keras).reshape(-1, lookback, 1)

# Make predictions
lstm_pred_scaled = lstm_model.predict(X_test_keras, verbose=0)
lstm_pred_test = scaler.inverse_transform(lstm_pred_scaled)[:TEST_SIZE]

lstm_rmse = np.sqrt(mean_squared_error(test_df['TTOKM'].values, lstm_pred_test))
lstm_mae = mean_absolute_error(test_df['TTOKM'].values, lstm_pred_test)
lstm_mape = np.mean(np.abs((test_df['TTOKM'].values - lstm_pred_test.flatten()) / test_df['TTOKM'].values)) * 100

print(f"\nLSTM Model - Test Set Performance:")
print(f"RMSE: {lstm_rmse:.2f}")
print(f"MAE: {lstm_mae:.2f}")
print(f"MAPE: {lstm_mape:.2f}%")

# Visualize LSTM forecast (training history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training history
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=1)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=1)
axes[0].set_title(f'{COMMODITY.upper()} - LSTM Training Loss', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=10)
axes[0].set_ylabel('Loss (MSE)', fontsize=10)
axes[0].legend(loc='best', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Test forecast
axes[1].plot(train_df['date'], train_df['TTOKM'], label='Training Data', linewidth=1, color='steelblue', alpha=0.7)
axes[1].plot(test_df['date'], test_df['TTOKM'], label='Test Data (Actual)', linewidth=1, color='darkred', alpha=0.8)
axes[1].plot(test_df['date'], lstm_pred_test, label='LSTM Forecast', linewidth=2, color='purple')

axes[1].set_title(f'{COMMODITY.upper()} - LSTM Model (52-week test forecast)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=10)
axes[1].set_ylabel('TTOKM (thousand ton-km)', fontsize=10)
axes[1].legend(loc='best', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create output directory
output_dir = f'./outputs/{COMMODITY}'
os.makedirs(output_dir, exist_ok=True)

# 1. Save forward forecast
forecast_export = forward_forecast.copy()
forecast_export.to_csv(f'{output_dir}/forecast_{FORECAST_WEEKS}weeks.csv', index=False)
print(f"✓ Saved: {output_dir}/forecast_{FORECAST_WEEKS}weeks.csv")

# 2. Save model metrics (test set performance)
metrics_df = pd.DataFrame({
    'Model': ['Prophet', 'SARIMAX', 'LSTM'],
    'RMSE': [prophet_rmse, sarimax_rmse, lstm_rmse],
    'MAE': [prophet_mae, sarimax_mae, lstm_mae],
    'MAPE_%': [prophet_mape, sarimax_mape, lstm_mape],
    'Best_Model': [
        'Yes' if best_mape_model == 'Prophet' else 'No',
        'Yes' if best_mape_model == 'SARIMAX' else 'No',
        'Yes' if best_mape_model == 'LSTM' else 'No'
    ]
})
metrics_df.to_csv(f'{output_dir}/model_metrics_test_set.csv', index=False)
print(f"✓ Saved: {output_dir}/model_metrics_test_set.csv")

# 3. Save test set comparison
test_comparison = pd.DataFrame({
    'date': test_df['date'].dt.strftime('%Y-%m-%d'),
    'actual': test_df['TTOKM'].values,
    'prophet': prophet_pred_test['yhat'].values,
    'prophet_lower': prophet_pred_test['yhat_lower'].values,
    'prophet_upper': prophet_pred_test['yhat_upper'].values,
    'sarimax': sarimax_pred_test[:TEST_SIZE] if sarimax_pred_test is not None else None,
    'lstm': lstm_pred_test.flatten()
})
test_comparison.to_csv(f'{output_dir}/test_set_forecasts.csv', index=False)
print(f"✓ Saved: {output_dir}/test_set_forecasts.csv")

# 4. Save commodity info
commodity_info = pd.DataFrame({
    'Commodity': [COMMODITY],
    'Total_Weeks': [len(df)],
    'Train_Weeks': [len(train_df)],
    'Test_Weeks': [len(test_df)],
    'Forecast_Weeks': [FORECAST_WEEKS],
    'Date_Start': [df['date'].min().strftime('%Y-%m-%d')],
    'Date_End': [df['date'].max().strftime('%Y-%m-%d')],
    'Best_Model': [best_mape_model],
    'Best_MAPE_%': [best_mape_value],
    'SARIMAX_Order_p_d_q': [str(sarimax_order) if sarimax_order else 'N/A'],
    'SARIMAX_Seasonal_P_D_Q_m': [str(sarimax_seasonal_order) if sarimax_seasonal_order else 'N/A']
})
commodity_info.to_csv(f'{output_dir}/commodity_info.csv', index=False)
print(f"✓ Saved: {output_dir}/commodity_info.csv")

print(f"\n{'='*60}")
print(f"✓ ALL RESULTS EXPORTED TO: {output_dir}/")
print(f"{'='*60}")

## 13. Export Results

Save models, forecasts, and metrics to CSV for thesis documentation.

In [ ]:
# Visualize forward forecast
fig, ax = plt.subplots(figsize=(15, 7))

# Historical data (last 2 years for context)
last_2_years = df[df['date'] >= df['date'].max() - pd.Timedelta(days=730)]

ax.plot(last_2_years['date'], last_2_years['TTOKM'], 
        label='Historical Data (Last 2 years)', linewidth=1.5, color='steelblue', alpha=0.8)

# Forward forecast
forecast_dates = pd.to_datetime(forward_forecast['date'])
ax.plot(forecast_dates, forward_forecast['forecast'], 
        label=f'Prophet Forecast ({FORECAST_WEEKS} weeks)', linewidth=2.5, color='green', marker='o', markersize=3)

# Confidence interval
ax.fill_between(forecast_dates, forward_forecast['lower_bound'], forward_forecast['upper_bound'], 
                alpha=0.25, color='green', label='95% Confidence Interval')

# Add vertical line at forecast start
ax.axvline(x=df['date'].max(), color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Forecast Start')

ax.set_title(f'{COMMODITY.upper()} - Forward Forecast ({FORECAST_WEEKS} weeks / {FORECAST_WEEKS/4.33:.1f} months)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('TTOKM (thousand ton-km)', fontsize=11)
ax.legend(loc='best', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Forward Forecast Visualization

Visualize 18-month forecast with confidence intervals.

In [ ]:
# Generate forward forecast using all available data
print(f"\n{'='*60}")
print(f"FORWARD FORECAST ({FORECAST_WEEKS} weeks ≈ {FORECAST_WEEKS/4.33:.1f} months ahead)")
print(f"{'='*60}")

# Prepare full dataset for Prophet
full_prophet_df = df[['date', 'TTOKM']].copy()
full_prophet_df.columns = ['ds', 'y']

# Retrain Prophet on full dataset for forward forecast
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    full_prophet_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10,
        seasonality_mode='additive'
    )
    full_prophet_model.fit(full_prophet_df)

# Generate future dates
future_dates = full_prophet_model.make_future_dataframe(periods=FORECAST_WEEKS, freq='W')
forward_forecast = full_prophet_model.predict(future_dates)

# Extract only forward forecast (exclude historical in-sample predictions)
forward_forecast = forward_forecast[-FORECAST_WEEKS:][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].reset_index(drop=True)
forward_forecast.columns = ['date', 'forecast', 'lower_bound', 'upper_bound']

# Convert dates to strings for export
forward_forecast['date'] = forward_forecast['date'].dt.strftime('%Y-%m-%d')

print(f"\nForecast Period: {forward_forecast['date'].iloc[0]} to {forward_forecast['date'].iloc[-1]}")
print(f"\nForward Forecast Summary:")
print(f"  Mean Forecast: {forward_forecast['forecast'].mean():.2f} TTOKM")
print(f"  Forecast Range: [{forward_forecast['lower_bound'].min():.2f}, {forward_forecast['upper_bound'].max():.2f}]")

print(f"\nFirst 10 weeks of forecast:")
print(forward_forecast.head(10).to_string(index=False))

## 11. Generate 18-Month Forecast

Create forward forecast on full dataset with specified forecast period (FORECAST_WEEKS).

In [ ]:
# Overlay all forecasts on test period
fig, ax = plt.subplots(figsize=(15, 7))

ax.plot(test_df['date'], test_df['TTOKM'], 
        label='Actual Test Data', linewidth=2.5, color='darkred', marker='o', markersize=4, zorder=5)

ax.plot(prophet_pred_test['ds'].dt.date, prophet_pred_test['yhat'], 
        label='Prophet', linewidth=2, color='green', linestyle='--', alpha=0.8)

if sarimax_pred_test is not None:
    ax.plot(test_df['date'][:len(sarimax_pred_test)], sarimax_pred_test, 
            label='SARIMAX', linewidth=2, color='orange', linestyle='--', alpha=0.8)

ax.plot(test_df['date'], lstm_pred_test, 
        label='LSTM', linewidth=2, color='purple', linestyle='--', alpha=0.8)

ax.set_title(f'{COMMODITY.upper()} - Model Forecast Comparison (52-week Test Period)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('TTOKM (thousand ton-km)', fontsize=11)
ax.legend(loc='best', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Test Set Visualization

Overlay all forecasts vs actual test period.

In [ ]:
# Model Comparison
results = {
    'Prophet': {'RMSE': prophet_rmse, 'MAE': prophet_mae, 'MAPE': prophet_mape},
    'SARIMAX': {'RMSE': sarimax_rmse, 'MAE': sarimax_mae, 'MAPE': sarimax_mape},
    'LSTM': {'RMSE': lstm_rmse, 'MAE': lstm_mae, 'MAPE': lstm_mape}
}

# Create comparison table
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.round(2)

print(f"\n{'='*60}")
print(f"MODEL COMPARISON - TEST SET METRICS")
print(f"{'='*60}")
print(comparison_df)
print(f"\n✓ Best Model (Lowest MAPE):")
best_mape_model = comparison_df['MAPE'].idxmin()
best_mape_value = comparison_df['MAPE'].min()
print(f"  {best_mape_model}: {best_mape_value:.2f}%")

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['RMSE', 'MAE', 'MAPE']
colors = ['steelblue', 'orange', 'purple']

for idx, metric in enumerate(metrics):
    models = comparison_df.index
    values = comparison_df[metric].values
    
    axes[idx].bar(models, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    axes[idx].set_title(f'{metric} Comparison', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(metric, fontsize=10)
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(values):
        if v is not None:
            axes[idx].text(i, v + 0.5, f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Model Comparison

Compare Prophet, SARIMAX, and LSTM performance on test set.

## 8. LSTM Model

Deep learning neural network for non-linear pattern capture.

In [ ]:
# SARIMAX Model with auto_arima
sarimax_rmse = None
sarimax_mae = None
sarimax_mape = None
sarimax_model = None
sarimax_order = None
sarimax_seasonal_order = None

try:
    print(f"\nSARIMAX Model - Tuning parameters...")
    print(f"{'='*50}")
    
    # Auto ARIMA with seasonal component (m=52 for yearly seasonality)
    auto_model = auto_arima(
        train_df['TTOKM'].values,
        start_p=0, max_p=3,
        start_q=0, max_q=3,
        start_d=0, max_d=2,
        start_P=0, max_P=2,
        start_Q=0, max_Q=2,
        m=52,  # Weekly seasonality (52 weeks per year)
        seasonal=True,
        stepwise=True,
        suppress_warnings=True,
        trace=False
    )
    
    sarimax_order = auto_model.order
    sarimax_seasonal_order = auto_model.seasonal_order
    
    print(f"Best ARIMA order: {sarimax_order}")
    print(f"Best Seasonal order: {sarimax_seasonal_order}")
    
    # Make predictions on test set
    sarimax_pred_test = auto_model.predict(n_periods=TEST_SIZE)
    
    sarimax_rmse = np.sqrt(mean_squared_error(test_df['TTOKM'].values, sarimax_pred_test[:len(test_df)]))
    sarimax_mae = mean_absolute_error(test_df['TTOKM'].values, sarimax_pred_test[:len(test_df)])
    sarimax_mape = np.mean(np.abs((test_df['TTOKM'].values - sarimax_pred_test[:len(test_df)]) / test_df['TTOKM'].values)) * 100
    
    print(f"\nSARIMAX Model - Test Set Performance:")
    print(f"RMSE: {sarimax_rmse:.2f}")
    print(f"MAE: {sarimax_mae:.2f}")
    print(f"MAPE: {sarimax_mape:.2f}%")
    
    # Visualize SARIMAX forecast
    fig, ax = plt.subplots(figsize=(14, 7))
    
    ax.plot(train_df['date'], train_df['TTOKM'], label='Training Data', linewidth=1, color='steelblue', alpha=0.7)
    ax.plot(test_df['date'], test_df['TTOKM'], label='Test Data (Actual)', linewidth=1, color='darkred', alpha=0.8)
    ax.plot(test_df['date'][:len(sarimax_pred_test)], sarimax_pred_test[:len(test_df)], 
            label='SARIMAX Forecast', linewidth=2, color='orange')
    
    ax.set_title(f'{COMMODITY.upper()} - SARIMAX Model (52-week test forecast)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('TTOKM (thousand ton-km)', fontsize=10)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    sarimax_model = auto_model
    
except Exception as e:
    print(f"✗ SARIMAX fitting failed: {e}")
    print(f"  Skipping SARIMAX model for this commodity")

## 7. SARIMAX Model

Statistical ARIMA approach with seasonal components (auto-tuned parameters).

In [ ]:
# Prophet model
# Prepare data for Prophet (columns: ds, y)
prophet_df = train_df[['date', 'TTOKM']].copy()
prophet_df.columns = ['ds', 'y']

# Train Prophet
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prophet_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10,
        seasonality_mode='additive'
    )
    prophet_model.fit(prophet_df)

# Make predictions on test set (52 weeks)
future_test = prophet_model.make_future_dataframe(periods=TEST_SIZE, freq='W')
prophet_forecast_test = prophet_model.predict(future_test)

# Extract forecasts for test period
prophet_pred_test = prophet_forecast_test[-TEST_SIZE:][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].reset_index(drop=True)

print(f"\nProphet Model - Test Set Performance:")
print(f"{'='*50}")
prophet_rmse = np.sqrt(mean_squared_error(test_df['TTOKM'].values, prophet_pred_test['yhat'].values))
prophet_mae = mean_absolute_error(test_df['TTOKM'].values, prophet_pred_test['yhat'].values)
prophet_mape = np.mean(np.abs((test_df['TTOKM'].values - prophet_pred_test['yhat'].values) / test_df['TTOKM'].values)) * 100

print(f"RMSE: {prophet_rmse:.2f}")
print(f"MAE: {prophet_mae:.2f}")
print(f"MAPE: {prophet_mape:.2f}%")

# Visualize Prophet forecast
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(train_df['date'], train_df['TTOKM'], label='Training Data', linewidth=1, color='steelblue', alpha=0.7)
ax.plot(test_df['date'], test_df['TTOKM'], label='Test Data (Actual)', linewidth=1, color='darkred', alpha=0.8)
ax.plot(prophet_pred_test['ds'].dt.date, prophet_pred_test['yhat'], label='Prophet Forecast', linewidth=2, color='green')
ax.fill_between(prophet_pred_test['ds'].dt.date, prophet_pred_test['yhat_lower'], prophet_pred_test['yhat_upper'], 
                 alpha=0.2, color='green', label='95% Confidence Interval')

ax.set_title(f'{COMMODITY.upper()} - Prophet Model (52-week test forecast)', fontsize=12, fontweight='bold')
ax.set_xlabel('Date', fontsize=10)
ax.set_ylabel('TTOKM (thousand ton-km)', fontsize=10)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Prophet Model

Facebook's Prophet model with multiple seasonalities and trend changes.

## 5. Train/Test Split

Prepare data for model validation: 95.8% training (~1199 weeks), 4.2% testing (~52 weeks / 1 year).

In [ ]:
# Augmented Dickey-Fuller (ADF) test for stationarity
from statsmodels.tsa.stattools import adfuller

result = adfuller(df['TTOKM'].dropna(), autolag='AIC')

print(f"\nAugmented Dickey-Fuller (ADF) Test Results:")
print(f"{'='*50}")
print(f"ADF Statistic: {result[0]:.6f}")
print(f"P-Value: {result[1]:.6f}")
print(f"Critical Values:")
for key, value in result[4].items():
    print(f"  {key}: {value:.3f}")

if result[1] <= 0.05:
    print(f"\n✓ Series is STATIONARY (p-value={result[1]:.6f} ≤ 0.05)")
    print(f"  Recommendation: d=0 (no differencing required)")
    d_recommend = 0
else:
    print(f"\n✗ Series is NON-STATIONARY (p-value={result[1]:.6f} > 0.05)")
    print(f"  Recommendation: d=1 or d=2 (differencing required)")
    d_recommend = 1

# Visualize original vs first difference
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df['date'], df['TTOKM'], linewidth=1, color='steelblue')
axes[0].set_title(f'{COMMODITY.upper()} - Original Series', fontsize=11, fontweight='bold')
axes[0].set_ylabel('TTOKM', fontsize=9)
axes[0].grid(True, alpha=0.3)

first_diff = df['TTOKM'].diff().dropna()
axes[1].plot(df['date'][1:], first_diff, linewidth=1, color='darkgreen')
axes[1].set_title(f'{COMMODITY.upper()} - First Difference (Differenced)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Δ TTOKM', fontsize=9)
axes[1].set_xlabel('Date', fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Stationarity Testing

Test if the series requires differencing (d parameter in SARIMAX).

In [ ]:
# Monthly aggregation for decomposition (weekly data has high frequency noise)
df_monthly = df.set_index('date').resample('ME')['TTOKM'].sum().reset_index()
df_monthly.columns = ['date', 'TTOKM']

# Decomposition (additive model suitable for transportation data)
try:
    decomposition = seasonal_decompose(df_monthly['TTOKM'], model='additive', period=52)
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    
    axes[0].plot(df_monthly['date'], df_monthly['TTOKM'], linewidth=1, color='steelblue')
    axes[0].set_title(f'{COMMODITY.upper()} - Observed (Monthly)', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('TTOKM', fontsize=9)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(df_monthly['date'], decomposition.trend, linewidth=1, color='darkgreen')
    axes[1].set_title(f'{COMMODITY.upper()} - Trend Component', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Trend', fontsize=9)
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(df_monthly['date'], decomposition.seasonal, linewidth=1, color='darkorange')
    axes[2].set_title(f'{COMMODITY.upper()} - Seasonal Component (Period=52 weeks)', fontsize=11, fontweight='bold')
    axes[2].set_ylabel('Seasonal', fontsize=9)
    axes[2].grid(True, alpha=0.3)
    
    axes[3].plot(df_monthly['date'], decomposition.resid, linewidth=1, color='darkred', alpha=0.7)
    axes[3].set_title(f'{COMMODITY.upper()} - Residual Component', fontsize=11, fontweight='bold')
    axes[3].set_ylabel('Residual', fontsize=9)
    axes[3].set_xlabel('Date', fontsize=9)
    axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nDecomposition Analysis:")
    print(f"Trend Variance: {decomposition.trend.var():.0f}")
    print(f"Seasonal Variance: {decomposition.seasonal.var():.0f}")
    print(f"Residual Variance: {decomposition.resid.var():.0f}")
    
except Exception as e:
    print(f"Decomposition failed: {e}")

## 3. Time Series Decomposition

Separate trend, seasonal, and residual components to understand underlying patterns.